In [ ]:
import os
import sys
import h5py
import time
import tables
import logging
import warnings
import datetime
import numpy as np
import pandas as pd
import scipy.io as spio
from scipy import signal
from scipy import interpolate

def convertItem2String(element):
    ds = file[element]
    asciiArr = file.get(ds.name).value
    return ''.join(chr(i) for i in asciiArr)

def convertItem2Int(element):
    ds = file[element]
    asciiArr = file.get(ds.name).value
    return int(''.join(chr(i) for i in asciiArr))

def convertItem2Float(element):
    ds = file[element]
    asciiArr = file.get(ds.name).value
    return float(''.join(chr(i) for i in asciiArr))

def getDoNotUse(file):
    doNotUse = (file.get((file.get('/session/doNotUse')).name).value)
    doNotUseFin = []
    for i in range(np.shape(doNotUse)[0]):
        doNotUseFin.append(doNotUse[i][0])
    return doNotUseFin

def getRegionIndex(file):
    regionMap = (file.get(file.get('/session/CLU2RegionMapping').name).value)
    return regionMap[0].astype(int)

def getBehData(file, doNotUse):
    trialwiseCorrect = file.get('/session/trialCorrect')
    correctTrials = []
    for trialNum in range(len(trialwiseCorrect)):
        correctTrials.append((trialwiseCorrect[trialNum])[0])
    behData = file.get('/session/behData/textdata')
    behDataShape = np.shape(behData)
    convBehData = []
    convBehData.append(np.transpose(correctTrials))
    convBehData.append(list(map(convertItem2String,behData[1])))
    convBehData.append(list(map(convertItem2String,behData[4])))
    convBehData.append(list(map(convertItem2String,behData[2])))
    convBehData.append(list(map(convertItem2String,behData[3])))
    convBehData = np.asarray(convBehData,dtype=object).transpose()
    convBehDataFin = []
    for i in range(np.shape(convBehData)[0]):
        tempHolder = []
        for j in range(np.shape(convBehData)[1]):
            tempHolder.append(convBehData[i][j])
        convBehDataFin.append(tempHolder)
    return convBehDataFin

def reparseBehData(inputBehData):
    typeTrial = []
    corrTrial = []
    jourTrial = []
    for a,arr in enumerate(inputBehData):
        typeTrial.append(arr[1][0])
        corrTrial.append(int(arr[0]))
        jourTrial.append(f'{arr[3][0]}{arr[4][0]}')
    return typeTrial,corrTrial,jourTrial

def getTrialRawLFPAndRawTimes(file,doNotUse,padToCount):
    sessionLFP = file.get('/session/LFPdata/data')
    rawLFP = []
    for trialNum in range(len(sessionLFP)):
        if doNotUse[trialNum] != 3:
            trialLFPds = file[sessionLFP[trialNum][0]]
            trialLFP = file.get(trialLFPds.name).value
            if np.shape(trialLFP)[0] == 2:
                rawLFP.append([])
            else:
                channel = trialLFP[:]
                rawLFP.append(channel)
    sessionTime = file.get('/session/LFPdata/TS')
    times = []
    for tNum in range(len(sessionTime)):
        if doNotUse[tNum] != 3:
            trialTimeds = file[sessionTime[tNum][0]]
            trialTime = file.get(trialTimeds.name).value
            if np.shape(trialTime) == 2:
                times.append([])
            else:
                channelTimes = (trialTime[:]).flatten()
                times.append(channelTimes)
    for i in range(padToCount-len(rawLFP)):
        rawLFP.append(None)
    for i in range(padToCount-len(times)):
        times.append(None)
    return rawLFP,times

def getWPRawLFPAndRawTimes(file,doNotUse,padToCount):
    sessionLFP = file.get('/session/WP/LFPdata/data')
    rawLFP = []
    for trialNum in range(len(sessionLFP)):
        if doNotUse[trialNum] != 3:
            trialLFPds = file[sessionLFP[trialNum][0]]
            trialLFP = file.get(trialLFPds.name).value
            if np.shape(trialLFP)[0] == 2:
                rawLFP.append([])
            else:
                channel = trialLFP[:]
                rawLFP.append(channel)
    sessionTime = file.get('/session/WP/LFPdata/TS')
    times = []
    for tNum in range(len(sessionTime)):
        if doNotUse[tNum] != 3:
            trialTimeds = file[sessionTime[tNum][0]]
            trialTime = file.get(trialTimeds.name).value
            if np.shape(trialTime) == 2:
                times.append([])
            else:
                channelTimes = (trialTime[:]).flatten()
                times.append(channelTimes)
    for i in range(padToCount-len(rawLFP)):
        rawLFP.append(None)
    for i in range(padToCount-len(times)):
        times.append(None)
    return rawLFP,times

def getReferencedTrialIndices(times1,times2,typeOfTrial):
    cueIndices = []
    spaIndices = []
    cueNames = []
    spaNames = []
    trNames = []
    cueIndex = 0
    spaIndex = 0
    for tr in range(len(typeOfTrial)):
        currType = typeOfTrial[tr]
        currTime1 = np.asarray(times1[tr]).flatten()
        currTime2 = np.asarray(times2[tr]).flatten()
        lenRange1 = len(np.arange(currTime1[0],currTime1[-1]+5000,5000))
        lenRange2 = len(np.arange(currTime2[0],currTime2[-1]+5000,5000))
        if currType == 'C':
            cueIndices.append(cueIndex+lenRange1)
            cueIndex += lenRange1
            cueIndices.append(cueIndex+lenRange2)
            cueIndex += lenRange2
            cueNames.append('WP')
            cueNames.append('TR')
        else:
            spaIndices.append(spaIndex+lenRange1)
            spaIndex += lenRange1
            spaIndices.append(spaIndex+lenRange2)
            spaIndex += lenRange2
            spaNames.append('WP')
            spaNames.append('TR')
    return cueIndices,spaIndices,cueNames,spaNames

def quickParseKey(inputKeyValues,ignoreList):
    outputKeyValues = [None]
    for a,values in enumerate(inputKeyValues):
        if a not in ignoreList:
            outputKeyValues.append(values)
    return outputKeyValues[1:]

def digestData(file):
    doNotUse = getDoNotUse(file)
    deleteList = np.nonzero(np.asarray(doNotUse).flatten())[0]
    behData = getBehData(file, doNotUse)
    trialType,trialCorrect,trialJourney = reparseBehData(behData)
    trialLFP,trialTimes = getTrialRawLFPAndRawTimes(file,doNotUse,len(doNotUse))
    wpLFP,wpTimes = getWPRawLFPAndRawTimes(file,doNotUse,len(doNotUse))
    updTimesWP = quickParseKey(wpTimes,deleteList)
    updTimesTR = quickParseKey(trialTimes,deleteList)
    updTrialType = quickParseKey(trialType,deleteList)
    cueInds,spaInds,namesCue,namesSpa = getReferencedTrialIndices(updTimesWP,updTimesTR,updTrialType)
    return cueInds,spaInds,namesCue,namesSpa

def loadValuesFromKnownFile(modelingFile):
    mmat = spio.loadmat(modelingFile)
    currDict = {'regionIndex':mmat.get('regionIndex').flatten(),'unitFields':mmat.get('unitFields'),
                'connectionAll':mmat.get('connectionAll'),'connectionHPC':mmat.get('connectionHPC'),
                'connectionPFC':mmat.get('connectionPFC'),'modelValuesAll':mmat.get('modelValuesAll').flatten(),
                'modelValuesHPC':mmat.get('modelValuesHPC').flatten(),
                'modelValuesPFC':mmat.get('modelValuesPFC').flatten(),
               }
    return currDict

rawCue = ''
rawSpa = ''
rawPos = ''
savCue = ''
savSpa = ''
warnings.filterwarnings('ignore')
for aRoot,aDirs,aFiles in os.walk(rawPos):
    for aFile in aFiles:
        if aFile.endswith('.mat'):
            currRawPosFile = os.path.join(rawPos,aFile)
            currSavSpaFile = os.path.join(savSpa,aFile)
            currSavCueFile = os.path.join(savCue,aFile)
            file = h5py.File(currRawPosFile, 'r')
            indsCue,indsSpa,namesC,namesS = digestData(file)
            try:
                currRawSpaFile = os.path.join(rawSpa,aFile)
                spaDict = loadValuesFromKnownFile(currRawSpaFile)
                spaDict.update({'timingInds':indsSpa,'timingNames':namesS})
                saveVar = spio.savemat(currSavSpaFile,spaDict)
            except:
                aHolderVariable = 0
            try:
                currRawCueFile = os.path.join(rawCue,aFile)
                cueDict = loadValuesFromKnownFile(currRawCueFile)
                cueDict.update({'timingInds':indsCue,'timingNames':namesC})
                saveVar = spio.savemat(currSavCueFile,cueDict)
            except:
                aHolderVariable = 0
            print(aFile)
print('Done')
